
# Data Preprocessing

This notebook contains the steps for loading, cleaning, and preprocessing the PV dataset.

In [1]:
import os
print(os.getcwd())

d:\GitHub\Project-DACO\notebooks


In [2]:
# Import libraries
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Define the file path relative to the current notebook directory
file_path = os.path.join("..", "data", "raw", "pv_module_efficiency_dataset.csv")

# Check if the file exists
if not os.path.exists(file_path):
    raise FileNotFoundError(f"The file {file_path} does not exist. Please check the path and try again.")

# Load the dataset
data = pd.read_csv(file_path)

# Filter and preprocess data
filtered_data = data[
    ~(
        (data['hotspot'] == 'None') &
        (data['birddrop'] == 'None') &
        (data['soiling'] == 'None') &
        (data['junction_box'] == 'None') &
        (data['affected_area'] > 0)
    )
]

# Separate the input features and target variables (output)
X = filtered_data.drop(["expected_efficiency", "efficiency_level"], axis=1)  # Input features
y_efficiency = filtered_data['expected_efficiency']  # Continuous target
y_efficiency_level = filtered_data['efficiency_level']  # Categorical target

# Convert categorical variables to dummy variables
X = pd.get_dummies(X, columns=['pv_module_type', 'hotspot', 'birddrop', 'soiling', 'junction_box'])

# Split the data into training and testing sets
X_train, X_test, y_efficiency_train, y_efficiency_test, y_efficiency_level_train, y_efficiency_level_test = train_test_split(
    X, y_efficiency, y_efficiency_level, test_size=0.2, random_state=42
)
label_mapping = {'bad': 0, 'extremely_bad': 1, 'good': 2, 'moderate': 3}
# Map the labels to integers using the defined mapping
y_efficiency_level_train_int = [label_mapping[label] for label in y_efficiency_level_train]
y_efficiency_level_test_int = [label_mapping[label] for label in y_efficiency_level_test]



# Identify numerical and binary features
numeric_features = X.select_dtypes(include=['float64', 'int64']).columns  # Numeric columns
binary_features = X.select_dtypes(include=['uint8', 'bool']).columns  # Binary/one-hot encoded columns

# Scale only the numerical features
scaler = StandardScaler()
X_train_scaled_numeric = scaler.fit_transform(X_train[numeric_features])
X_test_scaled_numeric = scaler.transform(X_test[numeric_features])

# Convert scaled numerical features back to DataFrame with original column names
X_train_scaled_numeric = pd.DataFrame(X_train_scaled_numeric, columns=numeric_features, index=X_train.index)
X_test_scaled_numeric = pd.DataFrame(X_test_scaled_numeric, columns=numeric_features, index=X_test.index)

# Combine scaled numerical features with binary features
X_train_scaled = pd.concat([X_train_scaled_numeric, X_train[binary_features]], axis=1)
X_test_scaled = pd.concat([X_test_scaled_numeric, X_test[binary_features]], axis=1)


# Save preprocessed data
processed_data_dir = os.path.join("..", "data", "processed")
os.makedirs(processed_data_dir, exist_ok=True)

X_train_scaled.to_csv(os.path.join(processed_data_dir, "X_train_scaled.csv"), index=False)
X_test_scaled.to_csv(os.path.join(processed_data_dir, "X_test_scaled.csv"), index=False)
pd.DataFrame(y_efficiency_train).to_csv(os.path.join(processed_data_dir, "y_efficiency_train.csv"), index=False)
pd.DataFrame(y_efficiency_test).to_csv(os.path.join(processed_data_dir, "y_efficiency_test.csv"), index=False)
pd.DataFrame(y_efficiency_level_train_int).to_csv(os.path.join(processed_data_dir, "y_efficiency_level_train_int.csv"), index=False)
pd.DataFrame(y_efficiency_level_test_int).to_csv(os.path.join(processed_data_dir, "y_efficiency_level_test_int.csv"), index=False)
pd.DataFrame(y_efficiency_level_train).to_csv(os.path.join(processed_data_dir, "y_efficiency_level_train.csv"), index=False)
pd.DataFrame(y_efficiency_level_test).to_csv(os.path.join(processed_data_dir, "y_efficiency_level_test.csv"), index=False)
